# Modelamiento - Adult Income

Este notebook es para experimentación y prototipado de modelos. **No es código de producción.**

## Objetivo

Comparar diferentes algoritmos de clasificación:
- Logistic Regression
- Random Forest
- Gradient Boosting
- XGBoost

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report

In [ ]:
# Cargar datos y preprocesador
from pathlib import Path

repo_root = Path.cwd().parent
data_dir = repo_root / 'data' / 'raw'
artifacts_dir = repo_root / 'artifacts'

# Cargar features
if (data_dir / 'features.parquet').exists():
    X = pd.read_parquet(data_dir / 'features.parquet')
else:
    X = pd.read_csv(data_dir / 'features.csv')

# Cargar targets
y = pd.read_parquet(data_dir / 'targets.parquet')
y = y.iloc[:, 0]

# Cargar preprocesador
preprocessor = joblib.load(artifacts_dir / 'preprocessor.joblib')

# Transformar
X_processed = preprocessor.transform(X)

print(f"X shape: {X_processed.shape}")
print(f"y shape: {y.shape}")

In [ ]:
# Dividir datos
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

In [ ]:
# Comparar modelos
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = []

for name, model in models.items():
    print(f"\nEvaluando {name}...")
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1_macro', n_jobs=-1)
    
    # Entrenar y evaluar
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='macro')
    acc = accuracy_score(y_test, y_pred)
    
    results.append({
        'model': name,
        'f1_cv_mean': cv_scores.mean(),
        'f1_cv_std': cv_scores.std(),
        'f1_test': f1,
        'accuracy': acc
    })
    
    print(f"  F1 CV: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    print(f"  F1 Test: {f1:.4f}")
    print(f"  Accuracy: {acc:.4f}")

# Mostrar ranking
results_df = pd.DataFrame(results).sort_values('f1_test', ascending=False)
print("\n" + "="*60)
print("RANKING DE MODELOS")
print("="*60)
print(results_df.to_string(index=False))